In [ ]:
import hashlib
import hmac as hmac_lib
import secrets

# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")


---

# Module 6: Bitcoin Applications

**Where we are:** We built ECDSA from scratch — signing, verification, the
nonce danger, and how signatures become bytes. ECDSA is what Bitcoin used from
2009 to 2021. But it has quirks that made developers' lives harder. This module
covers what replaced it and where else ECC shows up in Bitcoin.

## 6.1 From ECDSA to Schnorr — why Bitcoin switched

ECDSA works, but it has three practical problems:

**1. Variable-length signatures.** DER encoding produces 71-73 bytes depending
on whether $r$ or $s$ need a padding byte. Every piece of Bitcoin software has
to handle this variability. Fee estimation is harder because transaction size
isn't fully predictable before signing.

**2. Malleability.** Given a valid signature $(r, s)$, anyone can flip it to
$(r, N-s)$ — also valid. This changes the transaction's hash without
invalidating it. Bitcoin added the low-S rule (BIP 62) as a patch, but the
underlying problem is that ECDSA has this ambiguity baked in.

**3. No linearity.** You can't combine ECDSA signatures. If Alice and Bob both
sign, you need two separate signatures in the transaction. For a 3-of-5
multisig, that's 3 signatures x ~72 bytes = ~216 bytes of signatures alone.

Schnorr signatures (invented by Claus-Peter Schnorr in 1989, patent expired
2008) fix all three:

| Problem | ECDSA | Schnorr |
|---------|-------|---------|
| Size | ~72 bytes (variable, DER) | 64 bytes (fixed) |
| Malleability | Requires low-S fix | None by design |
| Linearity | Not linear | **Linear** — signatures add up |
| Batch verify | One at a time | Verify many at once |

The linearity is the big deal. Because Schnorr is linear, multiple signers can
produce a **single signature** that looks identical to a single-signer
signature (this is what MuSig2 does). A 3-of-3 multisig becomes 64 bytes —
same as a single key spend. Nobody looking at the blockchain can tell the
difference. That's a massive privacy and efficiency win.

### How Schnorr signing works

The equation is dramatically simpler than ECDSA:

```
  Signing:                           Verification:
  ────────                           ─────────────
  k = random nonce                   e = H(R_x || P_x || m)
  R = k × G                         Check: s × G  =  R + e × P
  e = H(R_x || P_x || m)
  s = k + e·d  mod N                That's it. One equation.
  Signature: (R_x, s) = 64 bytes
```

Compare with ECDSA's signing: $s = k^{-1}(z + rd)$. Schnorr doesn't need the
modular inverse $k^{-1}$. It's just $s = k + e \cdot d$ — addition and
multiplication. Simpler equation, simpler proofs, fewer things to go wrong.

### Why verification works

Same intuition as ECDSA — the private key cancels out:

$$s \times G = (k + ed) \times G = kG + edG = R + eP \quad \checkmark$$

The verifier checks $s \times G \stackrel{?}{=} R + e \times P$. The $d$ is
hidden inside $P = dG$. Multiplying $P$ by the public challenge $e$ pulls out
just enough of $d$ to make the equation balance.

### Why linearity matters

If Alice has key $d_A$ and Bob has key $d_B$, their combined public key is
$P_{AB} = P_A + P_B = (d_A + d_B) \times G$. They can each produce a partial
signature ($s_A = k_A + e \cdot d_A$, $s_B = k_B + e \cdot d_B$) and add them:
$s = s_A + s_B = (k_A + k_B) + e(d_A + d_B)$. That's a valid Schnorr signature
for the combined key. With ECDSA, the $k^{-1}$ in the equation destroys this
linearity — you can't just add two ECDSA signatures together.

Bitcoin activated Schnorr with the **Taproot** soft fork (November 2021, block
709,632). Every P2TR key-path spend uses Schnorr.

In [22]:
# Simplified Schnorr signature (BIP340-like)

def tagged_hash(tag: str, data: bytes) -> bytes:
    """BIP340 tagged hash: SHA256(SHA256(tag) || SHA256(tag) || data)."""
    tag_hash = hashlib.sha256(tag.encode()).digest()
    return hashlib.sha256(tag_hash + tag_hash + data).digest()

def schnorr_sign(message: bytes, private_key: int) -> bytes:
    """Simplified BIP340 Schnorr signature."""
    P = scalar_mult(private_key, G)
    d = private_key if P.y % 2 == 0 else SECP_N - private_key
    
    # Deterministic nonce (simplified)
    aux = secrets.token_bytes(32)
    t = int.from_bytes(aux, 'big') ^ d
    k_bytes = tagged_hash("BIP0340/nonce", 
                          t.to_bytes(32, 'big') + P.x.to_bytes(32, 'big') + message)
    k = int.from_bytes(k_bytes, 'big') % SECP_N
    if k == 0:
        raise ValueError("k is zero")
    
    R = scalar_mult(k, G)
    if R.y % 2 != 0:
        k = SECP_N - k
        R = scalar_mult(k, G)
    
    e_bytes = tagged_hash("BIP0340/challenge",
                          R.x.to_bytes(32, 'big') + P.x.to_bytes(32, 'big') + message)
    e = int.from_bytes(e_bytes, 'big') % SECP_N
    
    s = (k + e * d) % SECP_N
    return R.x.to_bytes(32, 'big') + s.to_bytes(32, 'big')

def schnorr_verify(message: bytes, signature: bytes, pubkey_x: int) -> bool:
    """Simplified BIP340 Schnorr verification."""
    R_x = int.from_bytes(signature[:32], 'big')
    s = int.from_bytes(signature[32:], 'big')
    
    # Recover P (assume even y)
    y2 = (pow(pubkey_x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if y % 2 != 0:
        y = SECP_P - y
    P = Point(pubkey_x, y)
    
    e_bytes = tagged_hash("BIP0340/challenge",
                          R_x.to_bytes(32, 'big') + pubkey_x.to_bytes(32, 'big') + message)
    e = int.from_bytes(e_bytes, 'big') % SECP_N
    
    # Verify: s×G = R + e×P
    lhs = scalar_mult(s, G)
    
    # Recover R (assume even y)
    R_y2 = (pow(R_x, 3, SECP_P) + 7) % SECP_P
    R_y = mod_sqrt(R_y2, SECP_P)
    if R_y % 2 != 0:
        R_y = SECP_P - R_y
    R = Point(R_x, R_y)
    
    rhs = point_add(R, scalar_mult(e, P))
    return lhs == rhs

# Demo
d_schnorr = secrets.randbelow(SECP_N - 1) + 1
P_schnorr = scalar_mult(d_schnorr, G)
msg_schnorr = b"Taproot transaction"

sig_schnorr = schnorr_sign(msg_schnorr, d_schnorr)
valid_schnorr = schnorr_verify(msg_schnorr, sig_schnorr, P_schnorr.x)

print("=== Schnorr Signature (BIP340) ===")
print(f"Message: {msg_schnorr.decode()}")
print(f"Signature: {sig_schnorr.hex()[:40]}...")
print(f"  R_x (32 bytes) + s (32 bytes) = {len(sig_schnorr)} bytes total")
print(f"Verification: {valid_schnorr}  ✓")
print(f"\nCompare: ECDSA ≈72 bytes (DER),  Schnorr = 64 bytes (fixed)")

=== Schnorr Signature (BIP340) ===
Message: Taproot transaction
Signature: 56decbab75c329a8fce34a11519915db68513e1e...
  R_x (32 bytes) + s (32 bytes) = 64 bytes total
Verification: True  ✓

Compare: ECDSA ≈72 bytes (DER),  Schnorr = 64 bytes (fixed)


## 6.2 Signatures on the Wire — ECDSA vs Schnorr

In Module 5 we built DER encoding for ECDSA signatures (~72 bytes, variable).
Schnorr is simpler: fixed 64 bytes, just `R_x (32 bytes) || s (32 bytes)`.

| Format | Encoding | Size | Malleability |
|--------|----------|------|-------------|
| ECDSA | DER (variable) | ~71-73 bytes + sighash | Requires low-S fix (BIP 62) |
| Schnorr | Fixed `R_x \|\| s` | 64 bytes | None by design |

### Where does the signature live in a transaction?

A Bitcoin transaction is a byte stream with this layout:

```
 SegWit Transaction (BIP 141)
┌──────────┬────────┬──────┬────────┬─────────┬─────────┬──────────┐
│ version  │ marker │ flag │ inputs │ outputs │ witness │ locktime │
│ 4 bytes  │  0x00  │ 0x01 │  var   │   var   │   var   │ 4 bytes  │
└──────────┴────────┴──────┴────────┴─────────┴─────────┴──────────┘
```

The signature's location depends on the script type:

| Script type | Where the signature goes | What else is included |
|-------------|------------------------|----------------------|
| **P2PKH** (legacy) | `scriptSig` field of the input | compressed public key (33 bytes) |
| **P2WPKH** (SegWit v0) | `witness` field (scriptSig is empty) | compressed public key (33 bytes) |
| **P2TR key-path** (Taproot) | `witness` field — just the 64-byte sig | nothing else needed |
| **P2TR script-path** (Taproot) | `witness` field — sigs + script + control block | tapscript, internal key, parity bit |

Notice the progression: P2PKH puts the signature in the input data (counted
fully toward block weight). SegWit moved it to the witness (discounted weight).
Taproot made it smaller (64 bytes instead of ~72) AND eliminated the need to
include the public key separately for key-path spends.

```
 P2TR Key-Path Witness
┌───────────────┬──────────────┬──────────────────┐
│ stack items: 1│ item len: 64 │ signature (64 B)  │
│    0x01       │    0x40      │  R_x || s         │
└───────────────┴──────────────┴──────────────────┘
```

In [ ]:
def der_encode_integer(value: int) -> bytes:
    b = value.to_bytes((value.bit_length() + 7) // 8, 'big')
    if b[0] & 0x80:
        b = b'\x00' + b
    return bytes([0x02, len(b)]) + b

def der_encode_signature(r: int, s: int) -> bytes:
    r_der = der_encode_integer(r)
    s_der = der_encode_integer(s)
    body = r_der + s_der
    return bytes([0x30, len(body)]) + body

r_example = 0xDEADBEEF_CAFEBABE_12345678_9ABCDEF0_DEADBEEF_CAFEBABE_12345678_9ABCDEF0
s_example = 0xFEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210

print("=== ECDSA: DER Encoding ===")
der_sig = der_encode_signature(r_example, s_example)
der_with_sighash = der_sig + bytes([0x01])
print(f"DER + sighash: {len(der_with_sighash)} bytes (variable)")

print(f"\n=== Schnorr: Fixed Format ===")
schnorr_sig = r_example.to_bytes(32, 'big') + s_example.to_bytes(32, 'big')
print(f"R_x || s:      {len(schnorr_sig)} bytes (fixed)")

print(f"\n=== P2TR Key-Path Witness ===")
witness = bytes([0x01, 0x40]) + schnorr_sig
print(f"Witness: {len(witness)} bytes total")
print(f"  0x01 = 1 stack item")
print(f"  0x40 = 64 bytes")
print(f"  {schnorr_sig.hex()[:24]}...{schnorr_sig.hex()[-8:]} = signature")
print(f"\nSavings per input: ~{len(der_with_sighash) - len(schnorr_sig)} bytes"
      f" ({len(der_with_sighash)} → {len(schnorr_sig)})")

## 6.3 ECDH in Lightning Onion Routing

### The problem

Lightning payments are routed through a chain of nodes: Alice → Bob → Carol →
Dave. Each node forwards the payment to the next. But here's the privacy
problem: if every node can see the full route, then Bob knows that Alice is
paying Dave through Carol. That's terrible for privacy.

The solution is **onion routing** — the same idea as Tor. Alice wraps the
payment instructions in layers of encryption, one layer per hop. Each node
can only peel its own layer, which tells it "forward this to the next node."
It can't see who the original sender is, who the final recipient is, or how
many hops are left.

### Where ECC comes in

To encrypt each layer, Alice needs a shared secret with each hop — but she
can't do an interactive Diffie-Hellman exchange with each node (that would
reveal the route). Instead, she uses **ECDH** — the elliptic curve version
of one-sided Diffie-Hellman from Module 4.

Alice picks one ephemeral secret $e$ and computes a shared secret with each
hop using their published public key:

```
  Alice's ephemeral key: e (secret), E = e×G (public, sent with payment)

  For hop 1 (Bob, public key P₁):
    Shared secret:  S₁ = SHA256(e × P₁)
    Bob can compute: S₁ = SHA256(d₁ × E)  ← same result, ECDH

  For hop 2 (Carol, public key P₂):
    Shared secret:  S₂ = SHA256(e × P₂)
    Carol can compute: S₂ = SHA256(d₂ × E')  ← E' is blinded (see below)

  For hop 3 (Dave, public key P₃):
    Shared secret:  S₃ = SHA256(e × P₃)
    Dave can compute: S₃ = SHA256(d₃ × E'')
```

Why does $e \times P_i = d_i \times E$? Same reason Diffie-Hellman works —
commutativity:
$$e \times P_i = e \times (d_i \times G) = d_i \times (e \times G) = d_i \times E$$

### The blinding trick

If every hop sees the same ephemeral key $E$, Bob and Carol could collude and
realize they're on the same route. So after each hop, the ephemeral key is
**blinded** — multiplied by a factor derived from the shared secret:

$$E' = E \times \text{SHA256}(E \| S_i)$$

Each hop sees a different version of $E$. No two hops can link their views.

### From shared secret to encryption

Each shared secret $S_i$ is used to derive actual encryption keys:
- `rho` key → encrypts routing info (ChaCha20)
- `mu` key → authenticates the payload (HMAC)

Alice builds the onion by encrypting from the inside out: Dave's layer first,
then Carol's, then Bob's. When Bob peels his layer, he sees "forward to Carol"
and a still-encrypted blob. He can't peek further.

In [ ]:
import hmac as hmac_lib

def ecdh_shared_secret(my_private: int, their_public: Point) -> bytes:
    """ECDH: compute shared secret from private key and other party's public key."""
    shared_point = scalar_mult(my_private, their_public)
    compressed = serialize_compressed(shared_point)
    return hashlib.sha256(compressed).digest()

def generate_key(shared_secret: bytes, key_type: str) -> bytes:
    """Derive a specific key from shared secret (BOLT #4 key derivation)."""
    return hmac_lib.new(key_type.encode(), shared_secret, hashlib.sha256).digest()

def blind_ephemeral(ephemeral_pub: Point, shared_secret: bytes) -> Point:
    """Blind ephemeral key so next hop can't link it to previous hop."""
    pub_bytes = serialize_compressed(ephemeral_pub)
    blind_bytes = hashlib.sha256(pub_bytes + shared_secret).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    return scalar_mult(blind_factor, ephemeral_pub)

# Simulate 3-hop onion routing
print("=== Lightning Onion Routing (ECDH) ===")
print("Sender → Hop1 → Hop2 → Hop3 (recipient)")

# Each hop has a key pair
hops = []
for i in range(3):
    d_hop = secrets.randbelow(SECP_N - 1) + 1
    P_hop = scalar_mult(d_hop, G)
    hops.append({'private': d_hop, 'public': P_hop, 'name': f'Hop{i+1}'})

# Sender creates ephemeral key pair
e_priv = secrets.randbelow(SECP_N - 1) + 1
E = scalar_mult(e_priv, G)  # Ephemeral public key (sent with packet)
print(f"\nSender ephemeral E: {serialize_compressed(E).hex()[:20]}...")

# Sender computes all shared secrets
ephemeral = E
sender_secrets = []
for hop in hops:
    ss = ecdh_shared_secret(e_priv, hop['public'])
    rho = generate_key(ss, "rho")
    mu = generate_key(ss, "mu")
    sender_secrets.append({'ss': ss, 'rho': rho, 'mu': mu})
    
    # Blind for next hop
    blind_bytes = hashlib.sha256(serialize_compressed(ephemeral) + ss).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    e_priv = (e_priv * blind_factor) % SECP_N
    ephemeral = scalar_mult(e_priv, G)

# Each hop computes the same shared secret using its private key
E_current = E
for i, hop in enumerate(hops):
    hop_ss = ecdh_shared_secret(hop['private'], E_current)
    match = hop_ss == sender_secrets[i]['ss']
    print(f"\n{hop['name']}:")
    print(f"  Receives E: {serialize_compressed(E_current).hex()[:20]}...")
    print(f"  Computes: d×E = shared secret")
    print(f"  Shared secret matches sender's: {match}  ✓")
    print(f"  Derives: rho (encrypt), mu (HMAC)")
    
    # Blind E for next hop
    blind_bytes = hashlib.sha256(serialize_compressed(E_current) + hop_ss).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    E_current = scalar_mult(blind_factor, E_current)

print(f"\nKey insight: each hop sees a different E (blinding prevents linking).")
print(f"Nobody except the sender knows the full route.")

=== Lightning Onion Routing (ECDH) ===
Sender → Hop1 → Hop2 → Hop3 (recipient)

Sender ephemeral E: 03bbbd712ab7404b458c...

Hop1:
  Receives E: 03bbbd712ab7404b458c...
  Computes: d×E = shared secret
  Shared secret matches sender's: True  ✓
  Derives: rho (encrypt), mu (HMAC)

Hop2:
  Receives E: 02d9189edf3910d5f6e9...
  Computes: d×E = shared secret
  Shared secret matches sender's: True  ✓
  Derives: rho (encrypt), mu (HMAC)

Hop3:
  Receives E: 02e339cda147abf97ea8...
  Computes: d×E = shared secret
  Shared secret matches sender's: True  ✓
  Derives: rho (encrypt), mu (HMAC)

Key insight: each hop sees a different E (blinding prevents linking).
Nobody except the sender knows the full route.


---